# Demo: RDFS Inference and Proof Reconstruction

This notebook demonstrates the current end-to-end reasoning path at the RDFLib level:

- add asserted triples to an RDFLib `Dataset`
- let `RETEStore` materialize inferred triples through `RETEEngine`
- capture engine-native `DerivationRecord` values during inference
- reconstruct a `DirectProof` for one inferred triple

The explanation step is still somewhat manual: we inject a recording derivation logger into `RETEEngineFactory`, then pass those records to `DerivationProofReconstructor`.

In [1]:
from pprint import pprint

from rdflib import Dataset, URIRef
from rdflib.namespace import RDF, RDFS
from rdflib.plugins.stores.memory import Memory
from rdflibr.engine import (
    RDFS_RULES,
    DerivationLogger,
    DerivationProofReconstructor,
    DerivationRecord,
    RETEEngineFactory,
    TripleFact,
)
from rdflibr.engine.rete_store import RETEStore

In [2]:
from rdflib import Graph, Node

graph: Graph


def pretty(triple: tuple[Node, Node, Node]) -> tuple[str, str, str]:
    # NOTE: This is **NOT** the canonical way to pretty-print derivations / proofs
    # TODO: Replace this after `Proof rendering` feature is available.
    return tuple(graph.qname(n) for n in triple)


class RecordingLogger(DerivationLogger):
    def __init__(self) -> None:
        self.records: list[DerivationRecord] = []

    def record(self, record: DerivationRecord) -> None:
        if len(record.conclusions) == 1 and isinstance(
            (fact := record.conclusions[0]), TripleFact
        ):
            print(f"Recording {fact.kind} derivation: {pretty(fact.triple)}")
        self.records.append(record)


logger = RecordingLogger()
factory = RETEEngineFactory(rules=RDFS_RULES, derivation_logger=logger)
store = RETEStore(Memory(), factory)
dataset = Dataset(store=store)
graph = dataset.default_graph

In [3]:
alice = URIRef("urn:test:alice")
person = URIRef("urn:test:Person")
mammal = URIRef("urn:test:Mammal")
animal = URIRef("urn:test:Animal")

graph.add((alice, RDF.type, person))
graph.add((person, RDFS.subClassOf, mammal))
print("-" * 80)
graph.add((mammal, RDFS.subClassOf, animal))
print("-" * 80)

sorted(pretty(t) for t in graph.triples((alice, RDF.type, None)))

Recording triple derivation: ('ns1:alice', 'rdf:type', 'ns1:Mammal')
--------------------------------------------------------------------------------
Recording triple derivation: ('ns1:alice', 'rdf:type', 'ns1:Animal')
--------------------------------------------------------------------------------


[('ns1:alice', 'rdf:type', 'ns1:Animal'),
 ('ns1:alice', 'rdf:type', 'ns1:Mammal'),
 ('ns1:alice', 'rdf:type', 'ns1:Person')]

In [4]:
goal = TripleFact(context=graph.identifier, triple=(alice, RDF.type, animal))

print("Goal triple materialized:", goal.triple in graph)
print("Recorded derivations:", len(logger.records))

for record in logger.records:
    print(record.rule_id, [fact.triple for fact in record.conclusions])

Goal triple materialized: True
Recorded derivations: 2
ruleset='rdfs' rule_id='rdfs9' [(rdflib.term.URIRef('urn:test:alice'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('urn:test:Mammal'))]
ruleset='rdfs' rule_id='rdfs9' [(rdflib.term.URIRef('urn:test:alice'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('urn:test:Animal'))]


In [5]:
proof = DerivationProofReconstructor().reconstruct(goal, logger.records)

print("Verdict:", proof.verdict)
print("Top-level node kind:", proof.proof.node_kind)
pprint(proof.model_dump(mode="python"))

Verdict: proved
Top-level node kind: rule_application
{'context': rdflib.term.URIRef('urn:x-rdflib:default'),
 'goal': {'context': rdflib.term.URIRef('urn:x-rdflib:default'),
          'kind': 'triple',
          'triple': (rdflib.term.URIRef('urn:test:alice'),
                     rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
                     rdflib.term.URIRef('urn:test:Animal'))},
 'notes': None,
 'proof': {'conclusions': [{'context': rdflib.term.URIRef('urn:x-rdflib:default'),
                            'kind': 'triple',
                            'triple': (rdflib.term.URIRef('urn:test:alice'),
                                       rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
                                       rdflib.term.URIRef('urn:test:Animal'))}],
           'derivation': {'bindings': [{'name': 'a',
                                        'value': rdflib.term.URIRef('urn:test:alice')},
                              

## Notes

- The graph materialization is driven entirely through the RDFLib store integration path.
- The proof reconstruction currently uses derivation logs, not a public `graph.explain(...)` API.
- If multiple derivation records can justify the same goal, the current reconstructor chooses the shallowest matching derivation.